In [0]:
%python

# Getting state code from widget
state = dbutils.widgets.get("state")
print(f"Running for state: {state}")

In [0]:
%python
# Reading and transforming data from master customers_orders_silver table
## Tranforming order data from unix timestamp to date and adding is_large_order column

df = spark.sql(f"""
    SELECT *,
        TO_DATE(FROM_UNIXTIME(order_datetime)) AS order_date,
        CASE WHEN number_of_line_items > 2 THEN true ELSE false END AS is_large_order
    FROM db_retail.silver.customers_orders_silver
    WHERE state = '{state}'
""")

# Save to UC table, one per state specified in the loop:
table_name = f"db_retail.silver.customers_orders_{state.replace(' ', '_').lower()}_silver"
df.write.mode("overwrite").saveAsTable(table_name)